In [ ]:
import qiskit
import qiskit_ibm_runtime
import qiskit_aer
import numpy
import scipy
print(qiskit.__version__)
print(qiskit_ibm_runtime.__version__)
print(qiskit_aer.__version__)
print(numpy.__version__)
print(scipy.__version__)


In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter, SuzukiTrotter

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import (
    Session,
    SamplerV2,
    SamplerOptions,
    EstimatorV2,
    EstimatorOptions,
    Batch,
)
from qiskit_aer import AerSimulator

import numpy as np
from scipy.optimize import minimize, curve_fit
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from rustworkx.visualization import graphviz_draw

np.random.seed(42)  # For reproducibility


In [ ]:
# =============================================================================
# Local utilities (standalone: no utils/*.py imports)
# XXZ: H = J * sum_<i,j> (X_i X_j + Y_i Y_j + Delta * Z_i Z_j)
# =============================================================================

def create_xxz_hamiltonian(edges, num_qubits, J, Delta):
    """Build XXZ terms on the given edges (Pauli operators, Qiskit little-endian)."""
    op_list = []
    for i, j in edges:
        label_x = ["I"] * num_qubits
        label_x[num_qubits - 1 - i] = "X"
        label_x[num_qubits - 1 - j] = "X"
        op_list.append(("".join(label_x), J))

        label_y = ["I"] * num_qubits
        label_y[num_qubits - 1 - i] = "Y"
        label_y[num_qubits - 1 - j] = "Y"
        op_list.append(("".join(label_y), J))

        label_z = ["I"] * num_qubits
        label_z[num_qubits - 1 - i] = "Z"
        label_z[num_qubits - 1 - j] = "Z"
        op_list.append(("".join(label_z), J * Delta))

    if not op_list:
        return SparsePauliOp(["I" * num_qubits], coeffs=[0.0])
    return SparsePauliOp.from_list(op_list)


def initialize_state(num_qubits, num_spin=2):
    circuit = QuantumCircuit(num_qubits)
    for i in range(0, num_qubits - 1, num_spin):
        circuit.ry(np.pi, i)
        circuit.x(i + 1)
        circuit.h(i)
        circuit.cx(i, i + 1)
    return circuit


def Rxyz(params):
    """U = exp(-i * (theta_x XX + theta_y YY + theta_z ZZ)), params = [Y, Z, X]."""
    circuit = QuantumCircuit(2)
    circuit.cx(1, 0)
    circuit.rz(2 * params[1], 0)
    circuit.h(1)
    circuit.rz(2 * params[2] + np.pi / 2, 1)
    circuit.cx(1, 0)
    circuit.rz(-2 * params[0], 0)
    circuit.h(1)
    circuit.cx(1, 0)
    circuit.rx(np.pi / 2, 0)
    circuit.rx(-np.pi / 2, 1)
    return circuit


def Rxyz_ansatz(num_qubits, reps, params=None):
    reps = 2 * reps
    if params is None:
        params = [[Parameter(f"theta^{i}_{j}") for j in range(3)] for i in range(reps)]
    circuit = QuantumCircuit(num_qubits)
    circuit = circuit.compose(initialize_state(num_qubits))
    for j in range(reps):
        if j % 2 == 0:
            for i in range(1, num_qubits - 1, 2):
                circuit = circuit.compose(Rxyz(params[j]), [i, i + 1])
        else:
            for i in range(0, num_qubits - 1, 2):
                circuit = circuit.compose(Rxyz(params[j]), [i, i + 1])
    return circuit


def two_qubit_gate_errors_per_circuit_layout(circuit, backend):
    pair_list = []
    error_pair_list = []
    error_acc_pair_list = []
    two_qubit_gate_count = 0
    properties = backend.properties()
    for instruction in circuit.data:
        gate_name = instruction.operation.name
        qubits = [q._index for q in instruction.qubits]
        if gate_name in ("barrier", "measure"):
            continue
        if len(qubits) == 2:
            two_qubit_gate_count += 1
            error_pair = properties.gate_error(gate=gate_name, qubits=qubits)
            if qubits not in pair_list:
                pair_list.append(qubits)
                error_pair_list.append(error_pair)
                error_acc_pair_list.append(error_pair)
            else:
                pos = pair_list.index(qubits)
                error_acc_pair_list[pos] += error_pair
    acc_two_qubit_error = sum(error_acc_pair_list)
    return (
        acc_two_qubit_error,
        two_qubit_gate_count,
        pair_list,
        error_pair_list,
        error_acc_pair_list,
    )


def finding_best_seed(circuit, backend):
    min_err_acc_seed_loop = 100
    circuit_opt_best_seed = None
    best_seed_transpiler = None
    two_qubit_gate_count_seed_loop = None
    for seed_transpiler in range(0, 500):
        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=3,
            seed_transpiler=seed_transpiler,
            layout_method="sabre",
        )
        circuit_opt_seed = pm.run([circuit])[0]
        acc_total_error_seed, two_qubit_gate_count_seed, *_ = (
            two_qubit_gate_errors_per_circuit_layout(circuit_opt_seed, backend)
        )
        if min_err_acc_seed_loop >= acc_total_error_seed:
            min_err_acc_seed_loop = acc_total_error_seed
            circuit_opt_best_seed = circuit_opt_seed
            best_seed_transpiler = seed_transpiler
            two_qubit_gate_count_seed_loop = two_qubit_gate_count_seed
    return (
        circuit_opt_best_seed,
        best_seed_transpiler,
        min_err_acc_seed_loop,
        two_qubit_gate_count_seed_loop,
    )


def fold_circuit(circuit, scale_factor):
    if scale_factor == 1:
        return circuit.copy()
    n = (scale_factor - 1) // 2
    folded_qc = circuit.copy()
    inverse_qc = circuit.inverse()
    for _ in range(n):
        folded_qc = folded_qc.compose(inverse_qc)
        folded_qc = folded_qc.compose(circuit)
    return folded_qc


def linear_fit(x, a, b):
    return a * x + b


def exponential_fit(x, a, b, c):
    return a * np.exp(-b * x) + c


def perform_zne(estimator, circuit, params, hamiltonian, scale_factors, backend=None, extrapolator="linear"):
    energies = []
    bound_circuit = circuit.assign_parameters(params)
    for m in scale_factors:
        noisy_circuit = fold_circuit(bound_circuit, m)
        target_backend = backend
        if target_backend is None and hasattr(estimator, "mode"):
            target_backend = estimator.mode
        if target_backend is not None:
            noisy_circuit = transpile(noisy_circuit, backend=target_backend, optimization_level=0)
        pub = (noisy_circuit, hamiltonian)
        result = estimator.run([pub]).result()[0]
        evs_data = result.data.evs
        energy = float(evs_data) if np.ndim(evs_data) == 0 else evs_data[0]
        energies.append(energy)
    try:
        if extrapolator == "exponential" and len(scale_factors) >= 3:
            p0 = [energies[0] - energies[-1], 0.1, energies[-1]]
            popt, _ = curve_fit(exponential_fit, scale_factors, energies, p0=p0, maxfev=2000)
            zero_noise_val = exponential_fit(0, *popt)
        else:
            popt, _ = curve_fit(linear_fit, scale_factors, energies)
            zero_noise_val = linear_fit(0, *popt)
    except Exception as e:
        print(f"  [Warning] {extrapolator} fit failed ({e}). Falling back to linear.")
        popt, _ = curve_fit(linear_fit, scale_factors, energies)
        zero_noise_val = linear_fit(0, *popt)
    return zero_noise_val, energies


def run_rzne(
    ansatz,
    optimal_params,
    hamiltonian,
    estimator,
    scale_factors=[1, 3, 5],
    ref_exact_energy=None,
    backend=None,
    extrapolator="linear",
    reps=1,
):
    print(f"\n=== Running rZNE Post-processing ({extrapolator} fit) ===")
    ref_params = np.zeros_like(optimal_params)
    if ref_exact_energy is None:
        print("  > Calculating Reference Exact Energy internally...")
        ref_circuit = ansatz.assign_parameters(ref_params)
        ref_exact_energy = Statevector(ref_circuit).expectation_value(hamiltonian).real
    print(f"  > Reference Exact Energy: {ref_exact_energy:.6f}")
    ref_zne_list = []
    ans_zne_list = []
    for i in range(1, reps + 1):
        print(f"{i}/{reps}  > 1/3 Performing ZNE on Reference State...")
        ref_zne, ref_hist = perform_zne(
            estimator, ansatz, ref_params, hamiltonian, scale_factors, backend=backend, extrapolator=extrapolator
        )
        ref_zne_list.append(ref_zne)
        print(f"{i}/{reps}  > 2/3 Performing ZNE on Optimal Ansatz State...")
        ans_zne, ans_hist = perform_zne(
            estimator, ansatz, optimal_params, hamiltonian, scale_factors, backend=backend, extrapolator=extrapolator
        )
        ans_zne_list.append(ans_zne)
    ref_zne = np.mean(ref_zne_list)
    ans_zne = np.mean(ans_zne_list)
    rescale_factor = 1.0 if abs(ref_zne) < 1e-9 else ref_exact_energy / ref_zne
    final_mitigated_energy = ans_zne * rescale_factor
    print("  > 3/3 Calculation Complete.")
    print(f"    - Ref Exact: {ref_exact_energy:.4f}")
    print(f"    - Ref ZNE  : {ref_zne:.4f} (Factor: {rescale_factor:.4f})")
    print(f"    - Raw Opt  : {ans_hist[0]:.4f} (m=1)")
    print(f"    - ZNE Opt  : {ans_zne:.4f}")
    print(f"    - rZNE Final: {final_mitigated_energy:.4f}")
    return final_mitigated_energy


def prepare_zne_pubs(circuit, params, hamiltonian, scale_factors, state_label, rep_idx, backend=None):
    pubs = []
    metadata = []
    bound_circuit = circuit.assign_parameters(params)
    for m in scale_factors:
        noisy_circuit = fold_circuit(bound_circuit, m)
        if backend is not None:
            noisy_circuit = transpile(noisy_circuit, backend=backend, optimization_level=0)
        pubs.append((noisy_circuit, hamiltonian))
        metadata.append({"label": state_label, "scale": m, "rep": rep_idx})
    return pubs, metadata


def run_rzne_batch(
    ansatz,
    optimal_params,
    hamiltonian,
    backend,
    scale_factors=[1, 3, 5],
    ref_exact_energy=None,
    extrapolator="linear",
    reps=1,
):
    print(f"\n=== Running rZNE Post-processing with Batch ({extrapolator} fit) ===")
    ref_params = np.zeros_like(optimal_params)
    if ref_exact_energy is None:
        print("  > Calculating Reference Exact Energy internally...")
        ref_circuit = ansatz.assign_parameters(ref_params)
        ref_exact_energy = Statevector(ref_circuit).expectation_value(hamiltonian).real
    print(f"  > Reference Exact Energy: {ref_exact_energy:.6f}")
    print("  > 조립 중: 모든 ZNE 회로를 생성하고 있습니다...")
    all_pubs = []
    all_metadata = []
    for i in range(1, reps + 1):
        ref_pubs, ref_meta = prepare_zne_pubs(ansatz, ref_params, hamiltonian, scale_factors, "ref", i, backend)
        all_pubs.extend(ref_pubs)
        all_metadata.extend(ref_meta)
        ans_pubs, ans_meta = prepare_zne_pubs(ansatz, optimal_params, hamiltonian, scale_factors, "ans", i, backend)
        all_pubs.extend(ans_pubs)
        all_metadata.extend(ans_meta)
    print(f"  > 총 {len(all_pubs)}개의 회로가 준비되었습니다.")
    print("  > Batch 실행: 하드웨어에 일괄 전송 중입니다...")
    with Batch(backend=backend) as batch:
        estimator = EstimatorV2(mode=batch)
        estimator.options.resilience_level = 1
        job = estimator.run(all_pubs)
        result = job.result()
    print("  > 측정 완료. 데이터를 분석합니다.")
    data_store = {rep: {"ref": {}, "ans": {}} for rep in range(1, reps + 1)}
    for idx, pub_result in enumerate(result):
        meta = all_metadata[idx]
        evs_data = pub_result.data.evs
        energy = float(evs_data) if np.ndim(evs_data) == 0 else float(evs_data[0])
        data_store[meta["rep"]][meta["label"]][meta["scale"]] = energy

    def extrapolate(energies):
        try:
            if extrapolator == "exponential" and len(scale_factors) >= 3:
                p0 = [energies[0] - energies[-1], 0.1, energies[-1]]
                popt, _ = curve_fit(exponential_fit, scale_factors, energies, p0=p0, maxfev=2000)
                return exponential_fit(0, *popt)
            popt, _ = curve_fit(linear_fit, scale_factors, energies)
            return linear_fit(0, *popt)
        except Exception:
            popt, _ = curve_fit(linear_fit, scale_factors, energies)
            return linear_fit(0, *popt)

    ref_zne_list, ans_zne_list, raw_opt_list = [], [], []
    for i in range(1, reps + 1):
        ref_energies = [data_store[i]["ref"][m] for m in scale_factors]
        ans_energies = [data_store[i]["ans"][m] for m in scale_factors]
        ref_zne_list.append(extrapolate(ref_energies))
        ans_zne_list.append(extrapolate(ans_energies))
        raw_opt_list.append(ans_energies[0])
    ref_zne = np.mean(ref_zne_list)
    ans_zne = np.mean(ans_zne_list)
    raw_opt_mean = np.mean(raw_opt_list)
    rescale_factor = 1.0 if abs(ref_zne) < 1e-9 else ref_exact_energy / ref_zne
    final_mitigated_energy = ans_zne * rescale_factor
    print("\n  > Calculation Complete.")
    print(f"    - Ref Exact: {ref_exact_energy:.4f}")
    print(f"    - Ref ZNE  : {ref_zne:.4f} (Factor: {rescale_factor:.4f})")
    print(f"    - Raw Opt  : {raw_opt_mean:.4f} (m=1)")
    print(f"    - ZNE Opt  : {ans_zne:.4f}")
    print(f"    - rZNE Final: {final_mitigated_energy:.4f}")
    return final_mitigated_energy


def plot_adiabatic_with_rzne(vqe_history, history_exact, step_boundaries, rzne_value, num_qubits):
    plt.figure(figsize=(12, 7))
    iters = range(len(vqe_history["min energy"]))
    final_iter = len(vqe_history["min energy"]) - 1
    plt.plot(
        iters,
        vqe_history["min energy"],
        label=f"VQE Energy Path: {vqe_history['min energy'][-1]:.4f}",
        color="tab:blue",
        linewidth=1.5,
        alpha=0.8,
    )
    plt.plot(
        iters,
        history_exact["gs"],
        "--",
        label=f"Exact Ground State: {history_exact['gs'][-1]:.4f}",
        color="navy",
        linewidth=2,
    )
    if "es1" in history_exact:
        plt.plot(
            iters,
            history_exact["es1"],
            ":",
            color="tab:red",
            alpha=0.4,
            label=f"1st Excited State: {history_exact['es1'][-1]:.4f}",
        )
    for x_pos in step_boundaries:
        plt.axvline(x=x_pos, color="gray", linestyle="-", alpha=0.2)
    plt.scatter(
        final_iter,
        rzne_value,
        color="limegreen",
        marker="*",
        s=250,
        zorder=10,
        label=f"rZNE Final Result: {rzne_value:.4f}",
    )
    last_step_start = step_boundaries[-1] if step_boundaries else 0
    plt.hlines(
        y=rzne_value,
        xmin=last_step_start,
        xmax=final_iter,
        colors="limegreen",
        linestyles="-.",
        linewidth=2,
        alpha=0.9,
    )
    plt.title(f"Adiabatic VQE (N={num_qubits})")
    plt.xlabel("Iterations")
    plt.ylabel("Energy")
    plt.legend(loc="upper right", framealpha=0.9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_adiabatic_comparison(s_vals, vqe_vals, rzne_vals, exact_dict):
    plt.figure(figsize=(10, 7))
    plt.plot(s_vals, exact_dict["es1"], ":", color="gray", alpha=0.5, label="Excited State")
    plt.plot(
        s_vals,
        exact_dict["gs"],
        "--",
        color="black",
        linewidth=2,
        label=f"Exact Ground State: {exact_dict['gs'][-1]}",
    )
    plt.plot(s_vals, vqe_vals, "o-", color="tab:blue", label="Raw VQE (Noisy)", alpha=0.7)
    plt.plot(
        s_vals,
        rzne_vals,
        "*-",
        color="limegreen",
        linewidth=2.5,
        markersize=10,
        label=f"rZNE Mitigated: {rzne_vals[-1]}",
        zorder=10,
    )
    plt.title("Adiabatic Evolution: Raw VQE vs rZNE Mitigation")
    plt.xlabel("Adiabatic Parameter $s$")
    plt.ylabel("Energy")
    plt.legend(loc="best")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_error_and_fidelity(vqe_history, step_boundaries, num_qubits):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    iters = range(len(vqe_history["error"]))
    color1 = "tab:red"
    ax1.set_xlabel("Iterations", fontsize=12)
    ax1.set_ylabel("Relative Energy Error ($|\\frac{E_{VQE} - E_{exact}}{E_{exact}}|$)", color=color1, fontsize=12)
    error_label = f"Rel Error: {vqe_history['error'][-1]:.6e}" if vqe_history["error"] else "Relative Error"
    ax1.plot(iters, vqe_history["error"], color=color1, label=error_label, alpha=0.8, linewidth=1.5)
    ax1.tick_params(axis="y", labelcolor=color1)
    ax1.set_yscale("log")
    has_fidelity = "fidelity" in vqe_history and len(vqe_history["fidelity"]) > 0
    if has_fidelity:
        ax2 = ax1.twinx()
        color2 = "tab:blue"
        ax2.set_ylabel("Fidelity ($|\\langle \\psi_{exact} | \\psi_{VQE} \\rangle|^2$)", color=color2, fontsize=12)
        ax2.plot(
            iters,
            vqe_history["fidelity"],
            color=color2,
            label=f"Fidelity: {vqe_history['fidelity'][-1]:.4f}",
            alpha=0.8,
            linewidth=1.5,
        )
        ax2.tick_params(axis="y", labelcolor=color2)
        ax2.set_ylim(-0.05, 1.05)
    for x_pos in step_boundaries:
        if x_pos > 0:
            ax1.axvline(x=x_pos, color="gray", linestyle="--", alpha=0.5)
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    if has_fidelity:
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="center right", framealpha=0.9)
        plt.title(f"VQE Convergence: Relative Error & Fidelity (N={num_qubits})", fontsize=14)
    else:
        ax1.legend(lines_1, labels_1, loc="upper right", framealpha=0.9)
        plt.title(f"VQE Convergence: Relative Energy Error Only (N={num_qubits})", fontsize=14)
    plt.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
# 1. 파라미터 설정
# 1D XXZ: H = J * sum_i (X_i X_{i+1} + Y_i Y_{i+1} + Delta * Z_i Z_{i+1})
J = 1.0
Delta = 0.5
num_qubits = 12

# 2. 결합(Layout) 분리
# 전체 연결: [(0,1), (1,2), ..., (N-2, N-1)]
full_layout = [(i, i + 1) for i in range(num_qubits - 1)]

# H_odd용 결합 (0-1, 2-3, ...) -> 서로 떨어져 있는 섬들
odd_edges = full_layout[0::2]

# H_even용 결합 (1-2, 3-4, ...) -> 섬들을 연결하는 다리들
even_edges = full_layout[1::2]

print(f"Odd Edges (Init): {odd_edges}")
print(f"Even Edges (Perturb): {even_edges}")
print(f"XXZ parameters: J={J}, Delta={Delta}")


# 4. H_odd와 H_even 생성
H_odd = create_xxz_hamiltonian(odd_edges, num_qubits, J, Delta)
H_even = create_xxz_hamiltonian(even_edges, num_qubits, J, Delta)

print("\n--- H_odd (Initial Problem) ---")
print(H_odd)

print("\n--- H_even (Perturbation) ---")
print(H_even)


In [ ]:
import numpy as np
from tenpy.networks.mps import MPS
from tenpy.models.spins import SpinModel
from tenpy.algorithms import dmrg

def solve_exact_dmrg(num_qubits, J, Delta, s=1, k=4):
    """
    eigsh(k=4)와 호환되는 형태의 결과를 반환하는 DMRG 래퍼.
    Pauli XXZ H = J * sum (XX + YY + Delta ZZ) 는 Spin-1/2 모델에서
    결합상수가 4배 (S = sigma/2) 이어야 에너지가 일치합니다.
    """
    num_bonds = num_qubits - 1
    Jx_arr = np.zeros(num_bonds)
    Jy_arr = np.zeros(num_bonds)
    Jz_arr = np.zeros(num_bonds)

    for i in range(num_bonds):
        weight = 4.0 * J if i % 2 == 0 else s * 4.0 * J
        Jx_arr[i] = weight
        Jy_arr[i] = weight
        Jz_arr[i] = weight * Delta

    model_params = {
        "L": num_qubits,
        "S": 0.5,
        "bc_MPS": "finite",
        "Jx": Jx_arr,
        "Jy": Jy_arr,
        "Jz": Jz_arr,
        "conserve": None,
    }
    model = SpinModel(model_params)

    dmrg_params = {
        "mixer": True,
        "trunc_params": {"chi_max": 150, "svd_min": 1.0e-10},
        "max_E_err": 1.0e-6,
        "max_sweeps": 10,
        "active_sites": 2,
    }

    exact_vals = []
    found_states = []

    print(f"    [DMRG] {k}개의 에너지 상태를 계산 중입니다...")
    for state_idx in range(k):
        initial_state = ["up" if np.random.rand() > 0.5 else "down" for _ in range(num_qubits)]
        psi = MPS.from_product_state(model.lat.mps_sites(), initial_state, bc="finite")
        if len(found_states) > 0:
            dmrg_params["orthogonal_to"] = [{"mps": st, "weight": 10.0} for st in found_states]
        info = dmrg.run(psi, model, dmrg_params)
        exact_vals.append(info["E"])
        found_states.append(psi.copy())

    exact_vals = np.array(exact_vals)
    exact_vals.sort()
    exact_vecs = None
    return exact_vals, exact_vecs

# --- 실행 테스트 ---
hamiltonian = H_odd + H_even  # s=1로 가정
exact_val, exact_vecs = solve_exact_dmrg(num_qubits, J, Delta)
print(f"True GS Energy (by DMRG): {exact_val[0]:.6f}")


In [ ]:
reps = 3
circuit = Rxyz_ansatz(num_qubits, reps=reps)

circuit.draw("mpl", style="iqp", scale=0.6)


In [ ]:
service = QiskitRuntimeService()
backend = service.backend("ibm_yonsei")
backend_sim = AerSimulator().from_backend(backend)
# print(backend)


backend_sim.set_options(
    method='matrix_product_state',
    noise_model=None,
    matrix_product_state_max_bond_dimension=256,
    matrix_product_state_truncation_threshold=1e-6
)

print(backend_sim)
print("현재 시뮬레이션 방식:", backend_sim.options.method)
print(backend_sim.options) # 필요시 주석 해제하여 확인


In [ ]:
# pm = generate_preset_pass_manager(
#     target=backend_sim.target, optimization_level=3, seed_transpiler=42
# )
_, seed, _, _ = finding_best_seed(circuit, backend_sim)
pm = generate_preset_pass_manager(
    target=backend_sim.target, optimization_level=3, seed_transpiler=seed, layout_method="sabre"
)
print(f"Using transpiler seed: {seed}")


In [ ]:
isa_circuit = pm.run(circuit)
isa_hamiltonian = hamiltonian.apply_layout(layout=isa_circuit.layout)
# isa_circuit.draw("mpl", style="iqp", scale=0.6)


In [ ]:
from scipy.sparse.linalg import eigsh
from qiskit.quantum_info import Statevector
import numpy as np
from scipy.optimize import minimize

def init_vqe_history():
    global vqe_history
    global history_exact
    global iter_history_exact
    # error와 fidelity 리스트를 추가로 초기화합니다.
    vqe_history = {
        "iters": 0,
        "min energy": [],
        "error": [],
        "fidelity": [],
        "prev_vector": None
    }
    history_exact = {"gs": [], "es1": [], "es2": [], "es3": []}
    iter_history_exact = {"gs": [], "es1": [], "es2": [], "es3": []}

def cost_func(params, ansatz, hamiltonian, estimator, exact_gs_state=None, logical_circuit=None, exact_energy=None):
    pub = (ansatz, hamiltonian, [params])
    result = estimator.run(pubs=[pub]).result()

    # 결과 파싱 (스칼라 및 배열 처리)
    evs_data = result[0].data.evs
    energy = float(evs_data) if np.ndim(evs_data) == 0 else float(evs_data[0])

    vqe_history["iters"] += 1
    vqe_history["min energy"].append(energy)

    # 변수 안전 초기화
    error = None
    fidelity = None

    # 1. [핵심 수정] 상대 오차(Relative Error) 계산
    if exact_energy is not None:
        if abs(exact_energy) > 1e-10: # 정답 에너지가 0이 아닐 때 정상적인 상대 오차 계산
            error = abs((energy - exact_energy) / exact_energy)
        else: # 혹시라도 에너지가 0 근처일 경우 절대 오차로 우회 (안전장치)
            error = abs(energy - exact_energy)

        vqe_history["error"].append(error)

    # 2. Fidelity 계산 (DMRG 환경에서는 생략됨)
    if exact_gs_state is not None and logical_circuit is not None:
        sv = Statevector(logical_circuit.assign_parameters(params))
        fidelity = np.abs(np.vdot(exact_gs_state, sv.data))**2
        vqe_history["fidelity"].append(fidelity)

    # 3. 로그 출력 (단어 'Rel Error'로 변경)
    log_str = f"Iters. done: {vqe_history['iters']:04d} [Energy: {energy:.6f}"
    if error is not None:
        log_str += f" | Rel Error: {error:.6e}" # 과학적 표기법으로 세밀한 변화 관찰
    if fidelity is not None:
        log_str += f" | Fidelity: {fidelity:.4f}"
    log_str += "]"

    print(log_str)

    return energy


In [ ]:
from scipy.sparse.linalg import eigsh
from qiskit.quantum_info import Statevector
from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
from scipy.optimize import minimize
import numpy as np

# 파라미터 초기화
noise_scale = 0.2
x0 = np.random.normal(0, noise_scale, isa_circuit.num_parameters)
print(x0)

# 시뮬레이터 세팅
estimator = AerEstimatorV2()

# --- Adiabatic VQE 실행부 ---
# --- 1. 저장할 리스트 초기화 ---
init_vqe_history()

s_history = []
vqe_final_energies = []
rzne_energies = []

# Reference Parameter (항상 Singlet/Identity 상태인 0으로 고정)
ref_params = np.zeros(isa_circuit.num_parameters)

# --- 2. Adiabatic Loop ---
s_list = [1]
step_boundaries = []
print("--- Starting Adiabatic VQE with DMRG and Step-wise rZNE ---")

for i, s in enumerate(s_list):
    # (1) 현재 Hamiltonian 정의
    current_hamiltonian = H_odd + s * H_even
    current_isa_hamiltonian = current_hamiltonian.apply_layout(isa_circuit.layout)

    # (2) [정답 계산] ED(eigsh) 대신 DMRG (k=4 호환 래퍼) 사용!
    start_iter_count = len(vqe_history["min energy"])
    step_boundaries.append(start_iter_count)

    print(f"\nStep {i} (s={s:.2f}) | Running DMRG for Exact States...")

    # k=4 옵션으로 바닥 상태 1개 + 들뜬 상태 3개의 에너지를 모두 구합니다.
    exact_vals, exact_vecs = solve_exact_dmrg(num_qubits, J, Delta, s=s, k=4)
    exact_gs_val = exact_vals[0]

    print(f"  > True GS Energy (DMRG): {exact_gs_val:.6f}")

    # (3) [VQE 수행]
    res = minimize(
        cost_func,
        x0,
        # [핵심 수정] cost_func의 인자 개수에 맞게 None 두 개와 exact_gs_val을 전달합니다!
        args=(isa_circuit, current_isa_hamiltonian, estimator, None, None, exact_gs_val),
        method="COBYLA",
        options={"maxiter": 1000, "tol": 1e-5, "disp": False}
    )
    vqe_final_energies.append(res.fun)
    x0 = res.x # 파라미터 업데이트

    end_iter_count = len(vqe_history["min energy"])
    iters_in_step = end_iter_count - start_iter_count

    s_history.append(s)

    # (4) Plotting을 위한 History 저장 (DMRG로 구한 들뜬 상태 4개를 모두 저장)
    history_exact["gs"].append(exact_vals[0])
    history_exact["es1"].append(exact_vals[1])
    history_exact["es2"].append(exact_vals[2])
    history_exact["es3"].append(exact_vals[3])

    iter_history_exact["gs"].extend([exact_vals[0]] * iters_in_step)
    iter_history_exact["es1"].extend([exact_vals[1]] * iters_in_step)
    iter_history_exact["es2"].extend([exact_vals[2]] * iters_in_step)
    iter_history_exact["es3"].extend([exact_vals[3]] * iters_in_step)

    # (5) [핵심] 매 단계 rZNE 수행
    # Reference Energy (주의: 50큐비트로 넘어가면 Statevector 대신 해석적/DMRG 계산 필요)
    current_ref_energy = (
        Statevector(circuit.assign_parameters(ref_params))
        .expectation_value(current_hamiltonian)
        .real
    )

    # rZNE 실행
    mitigated_val = run_rzne_batch(
        ansatz=isa_circuit,
        optimal_params=res.x,
        hamiltonian=current_isa_hamiltonian,
        backend=backend_sim,
        scale_factors=[1, 3, 5],
        ref_exact_energy=current_ref_energy,
        extrapolator="linear", # 노이즈리스 환경이므로 linear로 고정하여 안정성 확보
        reps=10
    )
    rzne_energies.append(mitigated_val)
    print(f"   > VQE: {res.fun:.4f} | rZNE: {mitigated_val:.4f}")

print("Optimization Complete.")


In [ ]:
# 기존 그래프 출력
plot_adiabatic_with_rzne(vqe_history, iter_history_exact, step_boundaries, rzne_energies[-1], num_qubits)
plot_adiabatic_comparison(s_history, vqe_final_energies, rzne_energies, history_exact)

# [새로 추가된 Error & Fidelity 그래프 출력]
plot_error_and_fidelity(vqe_history, step_boundaries, num_qubits)


In [ ]:
mitigated_val = run_rzne(
        ansatz=isa_circuit,
        optimal_params=res.x,
        hamiltonian=current_isa_hamiltonian,
        estimator=estimator,
        scale_factors=[1, 3, 5],
        extrapolator="exponential",
        ref_exact_energy=current_ref_energy,
        backend=backend_sim,
        reps=10
    )
